In [ ]:
# ============================================================================
# Fabric metadata + lineage extraction across Lakehouses / Warehouses
# ----------------------------------------------------------------------------
# Runtime : Fabric PySpark notebook (Runtime 1.2+ / Spark 3.4+)
# Auth    : identity running the notebook (interactive user or notebook owner)
# Requires: a DEFAULT LAKEHOUSE attached, and it must be SCHEMA-ENABLED
#           (OUTPUT_SCHEMA below relies on custom schema support)
#
# Outputs (Delta tables in the attached lakehouse):
#   <OUTPUT_SCHEMA>.view_definitions  one row per view, full CREATE VIEW text
#   <OUTPUT_SCHEMA>.objects           one row per table AND view  -> graph nodes
#   <OUTPUT_SCHEMA>.dependencies      one row per reference       -> graph edges
#   optional: Files/<SQL_EXPORT_FOLDER>/<item>/<schema>/<view>.sql
#
# Design notes:
# - Definitions come from sys.sql_modules (nvarchar(max), exact original text).
#   INFORMATION_SCHEMA.VIEWS.VIEW_DEFINITION is nvarchar(4000) and truncates.
# - Lineage comes from sys.sql_expression_dependencies, so no SQL parsing.
# - Results are returned client-side and written with Spark, deliberately NOT
#   with T-SQL INSERT ... SELECT: the Fabric engine rejects queries mixing
#   system catalog views with user tables ("not supported in distributed
#   processing mode").
# - All queries join system views to system views only, which is permitted.
# - node_id uniqueness is enforced CASE-INSENSITIVELY in Cell 6, because the
#   Power BI / Analysis Services Tabular semantic model this feeds compares
#   and dictionary-encodes text case-insensitively -- two node_id values
#   differing only by case (e.g. "budget" vs "Budget") are silently collapsed
#   into one stored value the moment Direct Lake loads the table, regardless
#   of what the Delta table itself actually contains. See Cell 6 for detail.
#
# NOT_VERIFIED: not executed against a live Fabric tenant. First run with
# DRY_RUN = True and confirm Cell 4's endpoint list looks right.
# ============================================================================

# MARKDOWN ********************
# ## Cell 1 - dependencies
# Prefer a custom Fabric Environment with semantic-link-labs attached to the
# notebook. Inline %pip is disabled by default in pipeline runs, and it
# restarts the interpreter (losing variables), so it must stay in cell 1.

# CELL ********************

%pip install semantic-link-labs==0.15.2

# CELL ********************
# ## Cell 2 - configuration

from datetime import datetime, timezone

# Workspaces to scan. Names or GUIDs. None => the notebook's own workspace.
WORKSPACES = None

# Restrict to specific items (names or GUIDs). Empty list => all in scope.
LAKEHOUSE_ALLOWLIST: list[str] = []

# Also scan Fabric Warehouses in the same workspaces.
INCLUDE_WAREHOUSES = True

# Output
OUTPUT_SCHEMA = "metadata"
VIEWS_TABLE = "view_definitions"
OBJECTS_TABLE = "objects"
DEPENDENCIES_TABLE = "dependencies"
WRITE_MODE = "overwrite"          # "overwrite" = snapshot, "append" = history
SQL_EXPORT_FOLDER = "view_ddl"    # None to skip per-view .sql file export
MAX_PARALLEL_ENDPOINTS = 8        # concurrent SQL endpoint connections
DRY_RUN = False                   # True = collect and display, write nothing

# Naive UTC: Spark TimestampType does not want tz-aware datetimes.
RUN_TS = datetime.now(timezone.utc).replace(tzinfo=None)

# CELL ********************
# ## Cell 3 - metadata queries

# One row per view, with the full DDL.
VIEW_DDL_QUERY = """
SELECT s.name        AS schema_name,
       v.name        AS view_name,
       v.create_date AS created_at,
       v.modify_date AS modified_at,
       m.definition  AS view_definition
FROM sys.views AS v
INNER JOIN sys.schemas AS s     ON s.schema_id = v.schema_id
INNER JOIN sys.sql_modules AS m ON m.object_id = v.object_id
WHERE v.is_ms_shipped = 0
ORDER BY s.name, v.name;
"""

# One row per table AND view -> the nodes of the lineage graph.
OBJECTS_QUERY = """
SELECT s.name        AS schema_name,
       o.name        AS object_name,
       o.type_desc   AS object_type,      -- USER_TABLE / VIEW
       o.create_date AS created_at,
       o.modify_date AS modified_at
FROM sys.objects AS o
INNER JOIN sys.schemas AS s ON s.schema_id = o.schema_id
WHERE o.is_ms_shipped = 0
  AND o.type IN ('U', 'V')
ORDER BY s.name, o.name;
"""

# One row per by-name reference -> the edges of the lineage graph.
# COALESCE on referenced_database_name is the important bit: NULL means the
# reference is inside the same lakehouse, populated means cross-lakehouse.
# referenced_type = 'UNRESOLVED' means the target is missing or external.
DEPENDENCIES_QUERY = """
SELECT DB_NAME()                                       AS referencing_db,
       OBJECT_SCHEMA_NAME(d.referencing_id)            AS referencing_schema,
       OBJECT_NAME(d.referencing_id)                   AS referencing_object,
       src.type_desc                                   AS referencing_type,
       COALESCE(d.referenced_database_name, DB_NAME()) AS referenced_db,
       d.referenced_schema_name                        AS referenced_schema,
       d.referenced_entity_name                        AS referenced_object,
       COALESCE(tgt.type_desc, 'UNRESOLVED')           AS referenced_type,
       d.is_ambiguous                                  AS is_ambiguous
FROM sys.sql_expression_dependencies AS d
INNER JOIN sys.objects AS src ON src.object_id = d.referencing_id
LEFT  JOIN sys.objects AS tgt ON tgt.object_id = d.referenced_id
WHERE d.referenced_entity_name IS NOT NULL;
"""

# CELL ********************
# ## Cell 4 - discovery: enumerate lakehouses / warehouses

import sempy.fabric as fabric
import sempy_labs as labs


def resolve_workspaces(workspaces):
    """Return [(workspace_name, workspace_id)] for the configured scope."""
    if not workspaces:
        ws_id = fabric.get_workspace_id()
        return [(fabric.resolve_workspace_name(ws_id), ws_id)]

    resolved = []
    for ws in workspaces:
        ws_id = fabric.resolve_workspace_id(ws)
        resolved.append((fabric.resolve_workspace_name(ws_id), ws_id))
    return resolved


def discover_endpoints(workspaces, allowlist, include_warehouses):
    """Return [(workspace_name, workspace_id, item_name, item_type)] to scan."""
    wanted = {"Lakehouse"}
    if include_warehouses:
        wanted.add("Warehouse")

    allow = {a.lower() for a in allowlist}
    endpoints = []

    for ws_name, ws_id in resolve_workspaces(workspaces):
        items = fabric.list_items(workspace=ws_id)
        for _, row in items.iterrows():
            item_type = row["Type"]
            item_name = row["Display Name"]
            item_id = row["Id"]
            if item_type not in wanted:
                continue
            if allow and not ({item_name.lower(), str(item_id).lower()} & allow):
                continue
            endpoints.append((ws_name, ws_id, item_name, item_type))

    return endpoints


endpoints = discover_endpoints(WORKSPACES, LAKEHOUSE_ALLOWLIST, INCLUDE_WAREHOUSES)
print(f"Discovered {len(endpoints)} SQL endpoint(s) to scan:")
for ws_name, _, item_name, item_type in endpoints:
    print(f"  [{item_type:9}] {ws_name} / {item_name}")

# CELL ********************
# ## Cell 5 - extraction
#
# ConnectLakehouse / ConnectWarehouse handle the Entra token exchange and the
# ODBC connection. One query per endpoint per query type, so cost scales with
# the number of lakehouses, not the number of views.

from concurrent.futures import ThreadPoolExecutor

import pandas as pd

EXTRACTION_ERRORS: list[dict] = []


def run_everywhere(sql, label):
    """Run one query against every discovered endpoint. Returns a DataFrame."""

    def _one(endpoint):
        ws_name, ws_id, item_name, item_type = endpoint
        connector = (labs.ConnectWarehouse if item_type == "Warehouse"
                     else labs.ConnectLakehouse)
        try:
            with connector(item_name, workspace=ws_id, timeout=120) as conn:
                df = conn.query(sql)
        except Exception as exc:               # noqa: BLE001 - isolate per endpoint
            EXTRACTION_ERRORS.append({
                "query": label,
                "workspace_name": ws_name,
                "item_name": item_name,
                "item_type": item_type,
                "error": f"{type(exc).__name__}: {exc}",
            })
            return None

        if df is None or df.empty:
            return pd.DataFrame()

        df = df.copy()
        df.insert(0, "workspace_name", ws_name)
        df.insert(1, "workspace_id", str(ws_id))
        df.insert(2, "item_name", item_name)    # Bronze / Silver / Gold
        df.insert(3, "item_type", item_type)
        df["extracted_at_utc"] = RUN_TS
        return df

    with ThreadPoolExecutor(max_workers=MAX_PARALLEL_ENDPOINTS) as pool:
        frames = [f for f in pool.map(_one, endpoints)
                  if f is not None and not f.empty]

    out = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    print(f"{label:14}: {len(out)} row(s)")
    return out


views_pdf = run_everywhere(VIEW_DDL_QUERY, "view_definitions")
objects_pdf = run_everywhere(OBJECTS_QUERY, "objects")
deps_pdf = run_everywhere(DEPENDENCIES_QUERY, "dependencies")

for err in EXTRACTION_ERRORS:
    print(f"  FAILED [{err['query']}] {err['item_name']}: {err['error']}")

# A NULL definition on a real view means the querying identity lacks
# VIEW DEFINITION on that object. It is a permission gap, not an empty view.
if not views_pdf.empty:
    missing = views_pdf[views_pdf["view_definition"].isna()]
    if not missing.empty:
        print(f"\nWARNING: {len(missing)} view(s) returned a NULL definition "
              f"(insufficient VIEW DEFINITION permission):")
        print(missing[["item_name", "schema_name", "view_name"]]
              .to_string(index=False))

# CELL ********************
# ## Cell 6 - graph keys
#
# Three-part keys keep nodes unique across layers. Edges point
# source (referenced) -> target (referencing), i.e. table feeds view, which is
# the direction arrows should flow in the app.


def three_part(df, item_col, schema_col, object_col):
    return (df[item_col].astype(str) + "."
            + df[schema_col].astype(str) + "."
            + df[object_col].astype(str))


_collided_plain_ids: set[str] = set()

if not objects_pdf.empty:
    objects_pdf["node_id"] = three_part(
        objects_pdf, "item_name", "schema_name", "object_name")

    # ------------------------------------------------------------------
    # Guarantee node_id uniqueness -- CASE-INSENSITIVELY -- before it is
    # used as a graph key or written to Delta.
    #
    # A name can legitimately exist as both a TABLE and a VIEW in the same
    # schema under a case-sensitive (BIN2) collation (e.g. table "entity"
    # + view "Entity") -- confirmed live: 11 such pairs across two
    # lakehouses, every one a genuine table+view pair, none a real data
    # duplicate.
    #
    # sys.objects on the SQL endpoint correctly returns two DIFFERENT
    # strings for such a pair (e.g. "budget" vs "Budget"), so a plain
    # case-SENSITIVE duplicate check here finds nothing to fix. But the
    # semantic model this feeds is a Power BI / Analysis Services Tabular
    # model, and Tabular compares and dictionary-encodes text
    # CASE-INSENSITIVELY -- confirmed live: two node_id values that differ
    # only by case are silently collapsed into ONE stored value the moment
    # Direct Lake loads this table, regardless of what this table actually
    # contains. So node_id must be made unique under a CASE-INSENSITIVE
    # comparison, matching the coarser granularity the consuming semantic
    # model actually operates at -- an exact-string check alone is not
    # enough.
    #
    # Disambiguate ONLY the rows that actually collide (case-insensitively)
    # with an object_type suffix. Every other object keeps its plain
    # {item}.{schema}.{object} id unchanged.
    # ------------------------------------------------------------------
    _dupe_mask = objects_pdf["node_id"].str.lower().duplicated(keep=False)
    _collided_plain_ids = set(
        objects_pdf.loc[_dupe_mask, "node_id"].str.lower().unique())

    if _dupe_mask.any():
        print(f"\nWARNING: {int(_dupe_mask.sum())} objects_pdf row(s) share "
              f"{len(_collided_plain_ids)} case-insensitively duplicate "
              f"node_id value(s) -- Power BI/Direct Lake will collapse "
              f"these onto ONE stored value regardless of casing (usually "
              f"a table + its same-named view). Disambiguating with an "
              f"object_type suffix:")
        print(objects_pdf.loc[_dupe_mask, ["item_name", "schema_name",
                                           "object_name", "object_type",
                                           "node_id"]]
              .sort_values("node_id").to_string(index=False))

        objects_pdf.loc[_dupe_mask, "node_id"] = (
            objects_pdf.loc[_dupe_mask, "node_id"] + "#"
            + objects_pdf.loc[_dupe_mask, "object_type"])

        # If rows STILL collide case-insensitively after appending
        # object_type, two objects share item+schema+object+type -- a
        # genuine duplicate, not a table/view casing collision. Fail
        # loudly rather than write corrupted data; silently picking one
        # would hide a real bug.
        _still = objects_pdf["node_id"].str.lower().duplicated(keep=False)
        if _still.any():
            raise ValueError(
                f"{int(_still.sum())} objects_pdf row(s) still share a "
                f"node_id (case-insensitively) after appending object_type "
                f"-- this is a genuine duplicate (same item+schema+object+"
                f"type), not a table/view casing collision. Inspect before "
                f"writing:\n" + objects_pdf.loc[_still].to_string(index=False))

if not deps_pdf.empty:
    deps_pdf["source_id"] = three_part(
        deps_pdf, "referenced_db", "referenced_schema", "referenced_object")
    deps_pdf["target_id"] = three_part(
        deps_pdf, "item_name", "referencing_schema", "referencing_object")

    # The referencing object is always a real, scanned object in THIS item,
    # and referencing_type is always reliable (populated via an INNER JOIN,
    # never NULL) -- so if target_id landed in the collided set above,
    # disambiguate it the same way objects_pdf's node_id was disambiguated,
    # or edges into a disambiguated object would stop resolving.
    if _collided_plain_ids:
        _tgt_collided = deps_pdf["target_id"].str.lower().isin(_collided_plain_ids)
        deps_pdf.loc[_tgt_collided, "target_id"] = (
            deps_pdf.loc[_tgt_collided, "target_id"] + "#"
            + deps_pdf.loc[_tgt_collided, "referencing_type"])

    # Cross-layer edges are the interesting ones for a medallion lineage view.
    deps_pdf["is_cross_item"] = (
        deps_pdf["referenced_db"].str.lower() != deps_pdf["item_name"].str.lower())
    print(f"Cross-item edges: {int(deps_pdf['is_cross_item'].sum())} "
          f"of {len(deps_pdf)}")

    # ------------------------------------------------------------------
    # Resolve each edge BY NAME against everything the scan found.
    #
    # referenced_type from Cell 3 is unreliable ONLY for cross-database
    # references and must not be used as a filter there. It comes from a
    # LEFT JOIN on sys.sql_expression_dependencies.referenced_id, which SQL
    # Server leaves NULL for every CROSS-DATABASE reference -- so a valid
    # cross-lakehouse edge is labelled UNRESOLVED exactly like a genuinely
    # broken one. sys.objects is also permission filtered, so an object the
    # identity cannot see looks missing too.
    #
    # For a SAME-item reference, though, SQL Server itself already resolved
    # referenced_id at compile time, so referenced_type reflects a REAL
    # engine-verified resolution and can be trusted -- including to
    # disambiguate a same-name table/view collision (see below).
    #
    # MATCHING IS EXACT FIRST. Under a case-sensitive (BIN2) collation a
    # schema can hold a table "entity" AND a view "Entity" as two distinct
    # objects. Case-insensitive matching collapses them, silently drops one,
    # and points every edge at whichever survived -- so a downstream view
    # would show the TABLE as its source instead of the VIEW.
    #
    # A case-insensitive fallback is still needed, because
    # sys.sql_expression_dependencies records referenced names as WRITTEN in
    # the referencing view's SQL, which need not match the object's real
    # casing when the source environment is case-insensitive. That fallback
    # is only applied when exactly one candidate exists.
    # ------------------------------------------------------------------
    _inv_exact, _inv_ci = {}, {}
    if not objects_pdf.empty:
        for _node, _otype in zip(objects_pdf["node_id"],
                                 objects_pdf["object_type"]):
            _inv_exact[_node] = _otype
            # Group by the PLAIN (pre-disambiguation) name for the
            # case-insensitive fallback, so e.g. "entity" still finds
            # "Entity#VIEW" as a candidate.
            _plain = _node.split("#", 1)[0]
            _inv_ci.setdefault(_plain.lower(), set()).add(_node)
    _scanned_items = ({i.lower() for i in objects_pdf["item_name"].unique()}
                      if not objects_pdf.empty else set())

    def _resolve(row):
        """-> (resolved_type, exact node_id or None, match_kind)."""
        src = row.source_id
        if src in _inv_exact:                       # authoritative, unambiguous
            return _inv_exact[src], src, "EXACT"
        if src.lower() in _collided_plain_ids:
            # Two+ objects share this plain name (case-insensitively). For
            # a same-item reference, referenced_type is an engine-verified
            # resolution (see comment above) -- use it to pick the correct
            # disambiguated node_id instead of guessing.
            if row.referenced_type != "UNRESOLVED":
                disambiguated = f"{src}#{row.referenced_type}"
                if disambiguated in _inv_exact:
                    return _inv_exact[disambiguated], disambiguated, "EXACT"
            # Cross-item reference to a collided name: no reliable type
            # signal available. Guessing would silently attach the wrong
            # parent and corrupt the lineage.
            return "AMBIGUOUS_CASE", None, "AMBIGUOUS"
        cands = _inv_ci.get(src.lower(), set())
        if len(cands) == 1:
            only = next(iter(cands))
            return _inv_exact[only], only, "CASE_MISMATCH"
        if len(cands) > 1:
            # e.g. both "entity" and "Entity" exist: the reference as written
            # matches neither exactly, and guessing would invent lineage.
            return "AMBIGUOUS_CASE", None, "AMBIGUOUS"
        if row.referenced_db.lower() not in _scanned_items:
            return "OUT_OF_SCAN_SCOPE", None, "OUT_OF_SCOPE"
        return "NOT_FOUND", None, "NOT_FOUND"

    _res = [_resolve(r) for r in deps_pdf.itertuples(index=False)]
    deps_pdf["resolved_type"] = [r[0] for r in _res]
    # The exact node_id of the object actually referenced. Join lineage on
    # THIS, not on source_id: source_id is the name as written in the SQL.
    deps_pdf["source_node_id"] = [r[1] for r in _res]
    deps_pdf["match_kind"] = [r[2] for r in _res]

    print("\nEdge resolution (join lineage on source_node_id):")
    print(deps_pdf["resolved_type"].value_counts().to_string())
    print(deps_pdf["match_kind"].value_counts().to_string())
    _raw_unres = int((deps_pdf["referenced_type"] == "UNRESOLVED").sum())
    _really = int(deps_pdf["resolved_type"].isin(
        ["NOT_FOUND", "OUT_OF_SCAN_SCOPE", "AMBIGUOUS_CASE"]).sum())
    print(f"referenced_type said UNRESOLVED for {_raw_unres} edge(s); "
          f"only {_really} are actually unresolvable.")

    # Objects that differ ONLY by case (not the identical-string collision
    # handled above) are the trap this fallback logic guards against.
    if not objects_pdf.empty:
        _collide = {low: sorted(names) for low, names in _inv_ci.items()
                    if len(names) > 1}
        if _collide:
            print(f"\n{len(_collide)} name(s) exist in MORE THAN ONE casing. "
                  f"These are distinct objects under a case-sensitive "
                  f"collation, and any edge whose written casing matches "
                  f"neither is left AMBIGUOUS_CASE rather than guessed:")
            for low, names in sorted(_collide.items())[:20]:
                types = [f"{n} ({_inv_exact[n]})" for n in names]
                print(f"  {' | '.join(types)}")

if not views_pdf.empty:
    views_pdf["node_id"] = three_part(
        views_pdf, "item_name", "schema_name", "view_name")
    # view_definitions only ever contains views, so any collided plain id
    # found here belongs to the VIEW side of a table/view collision --
    # apply the same suffix objects_pdf's VIEW row got, or the app's join
    # from a view node to its SQL definition will silently break.
    if _collided_plain_ids:
        _view_collided = views_pdf["node_id"].str.lower().isin(_collided_plain_ids)
        if _view_collided.any():
            views_pdf.loc[_view_collided, "node_id"] = (
                views_pdf.loc[_view_collided, "node_id"] + "#VIEW")

# CELL ********************
# ## Cell 7 - persist to Delta

from pyspark.sql.types import (BooleanType, StringType, StructField,
                               StructType, TimestampType)

_STR = StringType()
_TS = TimestampType()
_BOOL = BooleanType()

VIEWS_SCHEMA = StructType([
    StructField("workspace_name", _STR), StructField("workspace_id", _STR),
    StructField("item_name", _STR), StructField("item_type", _STR),
    StructField("schema_name", _STR), StructField("view_name", _STR),
    StructField("node_id", _STR),
    StructField("created_at", _TS), StructField("modified_at", _TS),
    StructField("view_definition", _STR),
    StructField("extracted_at_utc", _TS),
])

OBJECTS_SCHEMA = StructType([
    StructField("workspace_name", _STR), StructField("workspace_id", _STR),
    StructField("item_name", _STR), StructField("item_type", _STR),
    StructField("schema_name", _STR), StructField("object_name", _STR),
    StructField("object_type", _STR), StructField("node_id", _STR),
    StructField("created_at", _TS), StructField("modified_at", _TS),
    StructField("extracted_at_utc", _TS),
])

DEPENDENCIES_SCHEMA = StructType([
    StructField("workspace_name", _STR), StructField("workspace_id", _STR),
    StructField("item_name", _STR), StructField("item_type", _STR),
    StructField("referencing_schema", _STR),
    StructField("referencing_object", _STR),
    StructField("referencing_type", _STR),
    StructField("referenced_db", _STR),
    StructField("referenced_schema", _STR),
    StructField("referenced_object", _STR),
    StructField("referenced_type", _STR),      # raw, unreliable -- see Cell 6
    StructField("resolved_type", _STR),        # name-resolved: use this one
    StructField("source_node_id", _STR),       # exact node_id of the source
    StructField("match_kind", _STR),           # EXACT / CASE_MISMATCH / ...
    StructField("source_id", _STR), StructField("target_id", _STR),
    StructField("is_cross_item", _BOOL), StructField("is_ambiguous", _BOOL),
    StructField("extracted_at_utc", _TS),
])


def to_spark_df(pdf, schema):
    """Coerce pandas dtypes so createDataFrame accepts all-NULL columns."""
    pdf = pdf.copy()
    for field in schema.fields:
        col = pdf[field.name]
        if isinstance(field.dataType, TimestampType):
            pdf[field.name] = pd.to_datetime(col, errors="coerce")
        elif isinstance(field.dataType, BooleanType):
            pdf[field.name] = col.fillna(False).astype(bool)
        else:
            pdf[field.name] = col.where(col.notna(), None).astype(object)
    return spark.createDataFrame(pdf[[f.name for f in schema.fields]],
                                 schema=schema)


def write_delta(pdf, schema, table):
    """Write one metadata table. Returns rows written."""
    if pdf.empty:
        print(f"{table:18}: nothing to write")
        return 0
    if DRY_RUN:
        print(f"{table:18}: DRY_RUN, {len(pdf)} row(s) not written")
        display(pdf.head(10))
        return 0

    sdf = to_spark_df(pdf, schema)
    (sdf.write
        .format("delta")
        .mode(WRITE_MODE)
        .option("overwriteSchema", "true")
        .saveAsTable(f"{OUTPUT_SCHEMA}.{table}"))
    print(f"{table:18}: wrote {len(pdf)} row(s)")
    return len(pdf)


if not DRY_RUN:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {OUTPUT_SCHEMA}")

write_delta(views_pdf, VIEWS_SCHEMA, VIEWS_TABLE)
write_delta(objects_pdf, OBJECTS_SCHEMA, OBJECTS_TABLE)
write_delta(deps_pdf, DEPENDENCIES_SCHEMA, DEPENDENCIES_TABLE)

# CELL ********************
# ## Cell 8 - optional: one .sql file per view (for diffing / Git)

import re

import notebookutils


def safe_name(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]", "_", str(value))


if SQL_EXPORT_FOLDER and not views_pdf.empty and not DRY_RUN:
    base = f"Files/{SQL_EXPORT_FOLDER}"
    written = 0
    for row in views_pdf.itertuples(index=False):
        if row.view_definition is None or pd.isna(row.view_definition):
            continue
        path = (f"{base}/{safe_name(row.item_name)}"
                f"/{safe_name(row.schema_name)}"
                f"/{safe_name(row.view_name)}.sql")
        notebookutils.fs.put(path, row.view_definition, True)  # True = overwrite
        written += 1
    print(f"Exported {written} .sql file(s) under {base}")

# CELL ********************
# ## Cell 9 - sanity checks before building the app

if not objects_pdf.empty:
    print("Objects per layer / type:")
    print(objects_pdf.groupby(["item_name", "object_type"])
          .size().to_string())

if not deps_pdf.empty:
    # Report on resolved_type. referenced_type == UNRESOLVED is expected for
    # every cross-lakehouse edge and says nothing about validity.
    ambiguous = deps_pdf[deps_pdf["resolved_type"] == "AMBIGUOUS_CASE"]
    if not ambiguous.empty:
        print(f"\n{len(ambiguous)} edge(s) could not be resolved because the "
              f"referenced name exists in several casings and the written "
              f"casing matches none of them exactly:")
        print(ambiguous[["target_id", "source_id"]]
              .drop_duplicates().head(20).to_string(index=False))
        print("  Fix the casing in the source view definition; guessing here "
              "would attach the wrong parent and corrupt the lineage.")

    broken = deps_pdf[deps_pdf["resolved_type"] == "NOT_FOUND"]
    if not broken.empty:
        print(f"\n{len(broken)} reference(s) to objects that do NOT exist in "
              f"the scanned items -- broken views, or objects the identity "
              f"lacks permission to see:")
        print(broken[["target_id", "source_id"]]
              .drop_duplicates().head(20).to_string(index=False))

    outside = deps_pdf[deps_pdf["resolved_type"] == "OUT_OF_SCAN_SCOPE"]
    if not outside.empty:
        items = sorted(outside["referenced_db"].unique())
        print(f"\n{len(outside)} reference(s) point at item(s) that were not "
              f"scanned: {items}")
        print("  Add them to the scan (WORKSPACES / LAKEHOUSE_ALLOWLIST) or "
              "the lineage graph will have dangling edges.")

    # Views with no recorded dependency usually mean the reference is via a
    # OneLake shortcut, which SQL metadata cannot see as a cross-item edge.
    if not views_pdf.empty:
        no_deps = set(views_pdf["node_id"]) - set(deps_pdf["target_id"])
        # Note: a view can also appear here because its only references are
        # via OneLake shortcuts, which SQL metadata records as local.
        if no_deps:
            print(f"\n{len(no_deps)} view(s) with no tracked dependencies:")
            for n in sorted(no_deps)[:20]:
                print(f"  {n}")

# CELL ********************
# ## Cell 10 - optional: Spark-catalog views
#
# Views created through Spark SQL live in the lakehouse metastore, NOT in the
# SQL analytics endpoint, so Cell 5 cannot see them.

def spark_view_ddl(lakehouse_name: str) -> pd.DataFrame:
    rows = []
    schemas = [r.namespace for r in
               spark.sql(f"SHOW SCHEMAS IN {lakehouse_name}").collect()]
    for schema in schemas:
        for v in spark.sql(f"SHOW VIEWS IN {lakehouse_name}.{schema}").collect():
            name = v.viewName
            try:
                ddl = spark.sql(
                    f"SHOW CREATE TABLE {lakehouse_name}.{schema}.`{name}`"
                ).collect()[0][0]
            except Exception as exc:               # noqa: BLE001
                ddl = f"-- extraction failed: {type(exc).__name__}: {exc}"
            rows.append({
                "item_name": lakehouse_name,
                "item_type": "Lakehouse (Spark catalog)",
                "schema_name": schema,
                "view_name": name,
                "view_definition": ddl,
                "extracted_at_utc": RUN_TS,
            })
    return pd.DataFrame(rows)

# Example:
# display(spark_view_ddl("Silver_LH"))
